# 📊 기업별·기간별 재무분석 파이프라인 (Pandas)

- **담당자:** 안지형 (Pandas 재무분석 담당)
- **산출물:** `notebooks/analysis_anjihyeong.ipynb`
- **분석 목적:** 다양한 기업 재무 데이터를 입력받아 기간별 변화(매출액, 영업이익, 순이익, 자산 등)를 체계적으로 분석하고 SQL 산출물과 교차 검증을 수행하는 범용 분석 템플릿

> ⚠️ **[분석 및 해석 대원칙]**
> - 데이터에 존재하지 않는 외부 변수(경기, 경영진 의사결정 등)로 원인을 임의 추측하지 않습니다.
> - 오직 데이터에서 계산되고 관측된 객관적 사실(수치, 방향성, 순위)에 한정하여 결과를 도출합니다.

## 0. 환경 설정 및 라이브러리 임포트

In [1]:
import os
from pathlib import Path
import pandas as pd
import numpy as np

# 출력 서식 설정
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

print('Pandas 버전:', pd.__version__)

Pandas 버전: 3.0.5


## 1. 데이터 불러오기 및 설정

분석할 데이터 파일 경로와 필수 컬럼명을 설정합니다. (CSV 및 Excel 지원)

In [2]:
# =============================================================================
# [설정 영역] 파일 경로 및 주요 컬럼 정의 (사용자 환경에 맞게 변경 가능)
# =============================================================================
DATA_PATH = Path('../data/financial_data.csv') if Path('../data/financial_data.csv').exists() else Path('data/financial_data.csv')
SQL_RESULT_PATH = Path('../data/sql_result.csv') if Path('../data/sql_result.csv').exists() else Path('data/sql_result.csv')

# 분석 대상 필수 컬럼
COL_COMPANY = '기업명'
COL_PERIOD = '기간'
METRIC_COLS = ['매출액', '영업이익', '순이익', '자산']

def load_dataset(file_path):
    """파일 확장자에 맞춰 데이터를 안전하게 로드하는 범용 함수"""
    if file_path is None or not Path(file_path).exists():
        return None
    
    path = Path(file_path)
    if path.suffix.lower() == '.csv':
        return pd.read_csv(path)
    elif path.suffix.lower() in ['.xlsx', '.xls']:
        return pd.read_excel(path)
    else:
        raise ValueError(f'지원하지 않는 파일 형식입니다: {path.suffix}')

df_raw = load_dataset(DATA_PATH)
df_sql_raw = load_dataset(SQL_RESULT_PATH)

print(f'[로드 완료] 원천 데이터: {DATA_PATH} ({len(df_raw)}행 x {df_raw.shape[1]}열)')
if df_sql_raw is not None:
    print(f'[로드 완료] SQL 비교 데이터: {SQL_RESULT_PATH} ({len(df_sql_raw)}행 x {df_sql_raw.shape[1]}열)')
else:
    print('[알림] SQL 비교 파일이 지정되지 않았거나 없습니다.')

[로드 완료] 원천 데이터: data\financial_data.csv (16행 x 6열)
[로드 완료] SQL 비교 데이터: data\sql_result.csv (16행 x 6열)


## 2. 데이터 기본 확인 (요구사항 1)

데이터의 구조, 상위 샘플, 분석 대상 기업 및 기간 범위를 점검합니다.

In [3]:
def inspect_data(df, company_col=COL_COMPANY, period_col=COL_PERIOD):
    """데이터 기본 구조를 확인하고 요약 정보를 출력하는 함수"""
    print('=' * 60)
    print(' [1] 데이터 기본 구조 확인')
    print('=' * 60)
    print(f'- 전체 행 수: {df.shape[0]:,}개 / 전체 열 수: {df.shape[1]:,}개')
    print(f'- 포함된 컬럼: {df.columns.tolist()}')
    
    if company_col in df.columns:
        companies = df[company_col].dropna().unique().tolist()
        print(f'- 분석 대상 기업 ({len(companies)}개사): {companies}')
        
    if period_col in df.columns:
        periods = sorted(df[period_col].astype(str).unique().tolist())
        print(f'- 분석 대상 기간 ({len(periods)}개 기간): {periods}')
        
    print('\n--- [상위 5개 행 미리보기] ---')
    display(df.head())

inspect_data(df_raw)

 [1] 데이터 기본 구조 확인
- 전체 행 수: 16개 / 전체 열 수: 6개
- 포함된 컬럼: ['기업명', '기간', '매출액', '영업이익', '순이익', '자산']
- 분석 대상 기업 (4개사): ['삼성전자', 'SK하이닉스', '현대자동차', 'LG에너지솔루션']
- 분석 대상 기간 (4개 기간): ['2021', '2022', '2023', '2024']

--- [상위 5개 행 미리보기] ---


,기업명,기간,매출액,영업이익,순이익,자산
0,삼성전자,2021,2796048,516339,399074,4266212
1,삼성전자,2022,3022314,433766,556541,4484245
2,삼성전자,2023,2589355,65670,154871,4559060
3,삼성전자,2024,3050400,358000,325000,4800000
4,SK하이닉스,2021,429978,124103,96162,963854


## 3. 결측치 확인 (요구사항 2)

컬럼별 결측치 개수와 결측 비율(%)을 확인하고 결측 행을 진단합니다.

In [4]:
def check_missing_values(df):
    """컬럼별 결측치 및 결측 비율을 요약하는 함수"""
    missing_df = pd.DataFrame({
        '데이터타입': df.dtypes,
        '결측치_개수': df.isna().sum(),
        '결측률(%)': (df.isna().mean() * 100).round(2)
    })
    
    total_missing_rows = df.isna().any(axis=1).sum()
    
    print('=' * 60)
    print(' [2] 결측치 점검 결과')
    print('=' * 60)
    display(missing_df)
    print(f'- 결측치가 1개 이상 존재하는 행의 수: {total_missing_rows}개 (전체의 {total_missing_rows/len(df)*100:.2f}%)')
    return missing_df

_ = check_missing_values(df_raw)

 [2] 결측치 점검 결과
- 결측치가 1개 이상 존재하는 행의 수: 0개 (전체의 0.00%)


,데이터타입,결측치_개수,결측률(%)
기업명,str,0,0.00
기간,int64,0,0.00
매출액,int64,0,0.00
영업이익,int64,0,0.00
순이익,int64,0,0.00
자산,int64,0,0.00


## 4. 데이터 타입 확인 및 전처리 (요구사항 3)

필수 컬럼 존재 여부를 검증하고, 수치형 지표를 변환하며 기간 정렬을 수행합니다.

In [5]:
def preprocess_financial_data(df, company_col=COL_COMPANY, period_col=COL_PERIOD, metrics=METRIC_COLS):
    """필수 컬럼 검증, 수치형 변환 및 정렬을 수행하는 전처리 함수"""
    required = [company_col, period_col] + metrics
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise KeyError(f'필수 컬럼이 누락되었습니다: {missing}')
    
    df_clean = df.copy()
    
    # 1) 수치형 지표 변환 (문자열, 콤마 등 포함 시 처리)
    for col in metrics:
        if df_clean[col].dtype == object:
            df_clean[col] = df_clean[col].astype(str).str.replace(',', '', regex=False)
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')
        
    # 2) 기간 문자열 정규화
    df_clean[period_col] = df_clean[period_col].astype(str)
    
    # 3) 기업명 및 기간 기준 시계열 정렬
    df_clean = df_clean.sort_values(by=[company_col, period_col]).reset_index(drop=True)
    
    print('=' * 60)
    print(' [3] 전처리 및 데이터 타입 확인')
    print('=' * 60)
    print(df_clean[required].dtypes)
    
    print('\n--- [수치형 지표 기술 통계] ---')
    display(df_clean[metrics].describe())
    
    return df_clean

df_clean = preprocess_financial_data(df_raw)

 [3] 전처리 및 데이터 타입 확인
기업명       str
기간        str
매출액     int64
영업이익    int64
순이익     int64
자산      int64
dtype: object

--- [수치형 지표 기술 통계] ---


,매출액,영업이익,순이익,자산
count,16.00,16.00,16.00,16.00
mean,"1,270,788.12","140,859.19","130,623.50","2,144,982.31"
std,"1,078,927.55","166,536.70","168,377.62","1,663,587.69"
min,"178,519.00","-77,303.00","-91,375.00","237,641.00"
25%,"335,005.50","19,257.00","14,609.50","836,386.00"
50%,"919,003.00","84,132.50","87,999.00","1,746,946.50"
75%,"1,961,563.75","182,400.00","165,653.25","3,256,553.00"
max,"3,050,400.00","516,339.00","556,541.00","4,800,000.00"


## 5. 기간별 증감액 및 증감률 계산 (요구사항 4, 5)

동일 기업 내에서 직전 기간(Lag 1) 대비 **증감액**과 **증감률(%)**을 자동 계산합니다.

- **증감액** = $현재 기간 값 - 직전 기간 값$
- **증감률(%)** = $\frac{현재 기간 값 - 직전 기간 값}{|직전 기간 값|} \times 100$
- *예외 처리:* 직전 기간 값이 `0`이거나 결측치(`NaN`)인 경우 증감률은 정의할 수 없으므로 `NaN`으로 처리합니다.

In [6]:
def calculate_growth_metrics(df, company_col=COL_COMPANY, period_col=COL_PERIOD, metrics=METRIC_COLS):
    """동일 기업 내 시계열 증감액 및 증감률을 계산하는 함수"""
    res_df = df.copy()
    
    for col in metrics:
        # 직전 기간 값 산출 (동일 기업 그룹 기준)
        prev = res_df.groupby(company_col)[col].shift(1)
        
        # 1) 증감액
        res_df[f'{col}_증감액'] = res_df[col] - prev
        
        # 2) 증감률 (%) - 분모 0 및 결측치 예외 처리
        res_df[f'{col}_증감률(%)'] = np.where(
            prev.notna() & (prev != 0),
            (res_df[col] - prev) / prev.abs() * 100,
            np.nan
        )
        
    return res_df

df_analyzed = calculate_growth_metrics(df_clean)

print('=' * 60)
print(' [4 & 5] 기간별 증감액 및 증감률 계산 완료')
print('=' * 60)

# 요약 뷰 컬럼 구성
summary_cols = [COL_COMPANY, COL_PERIOD]
for m in METRIC_COLS:
    summary_cols.extend([m, f'{m}_증감액', f'{m}_증감률(%)'])

display(df_analyzed[summary_cols].head(10))

 [4 & 5] 기간별 증감액 및 증감률 계산 완료


,기업명,기간,매출액,매출액_증감액,매출액_증감률(%),영업이익,영업이익_증감액,영업이익_증감률(%),순이익,순이익_증감액,순이익_증감률(%),자산,자산_증감액,자산_증감률(%)
0,LG에너지솔루션,2021,178519,NaN,NaN,7685,NaN,NaN,9298,NaN,NaN,237641,NaN,NaN
1,LG에너지솔루션,2022,255986,"77,467.00",43.39,12132,"4,447.00",57.87,7751,"-1,547.00",-16.64,382994,"145,353.00",61.16
2,LG에너지솔루션,2023,337455,"81,469.00",31.83,21632,"9,500.00",78.31,16380,"8,629.00",111.33,453982,"70,988.00",18.54
3,LG에너지솔루션,2024,256200,"-81,255.00",-24.08,5800,"-15,832.00",-73.19,3400,"-12,980.00",-79.24,432000,"-21,982.00",-4.84
4,SK하이닉스,2021,429978,NaN,NaN,124103,NaN,NaN,96162,NaN,NaN,963854,NaN,NaN
5,SK하이닉스,2022,446481,"16,503.00",3.84,70067,"-54,036.00",-43.54,24389,"-71,773.00",-74.64,1038715,"74,861.00",7.77
6,SK하이닉스,2023,327657,"-118,824.00",-26.61,-77303,"-147,370.00",-210.33,-91375,"-115,764.00",-474.66,984024,"-54,691.00",-5.27
7,SK하이닉스,2024,661900,"334,243.00",102.01,234600,"311,903.00",403.48,198000,"289,375.00",316.69,1150000,"165,976.00",16.87
8,삼성전자,2021,2796048,NaN,NaN,516339,NaN,NaN,399074,NaN,NaN,4266212,NaN,NaN
9,삼성전자,2022,3022314,"226,266.00",8.09,433766,"-82,573.00",-15.99,556541,"157,467.00",39.46,4484245,"218,033.00",5.11


## 6. 기업별 비교 분석 (요구사항 6)

최신 분석 기간을 기준으로 각 기업의 재무 규모 순위 및 직전 대비 성장률을 상호 비교합니다.

In [7]:
def compare_companies(df, company_col=COL_COMPANY, period_col=COL_PERIOD, metrics=METRIC_COLS):
    """기업별 최신 기간 지표 비교 및 순위표를 생성하는 함수"""
    # 기업별 최신 기간 행 추출
    latest_df = df.groupby(company_col, as_index=False).tail(1).sort_values(by=company_col).reset_index(drop=True)
    
    print('=' * 60)
    print(' [6] 최신 기간 기업별 재무 지표 비교')
    print('=' * 60)
    display(latest_df[[company_col, period_col] + metrics])
    
    # 지표별 순위표
    print('\n--- [지표별 기업 순위] ---')
    for m in metrics:
        ranked = latest_df[[company_col, period_col, m]].dropna().sort_values(by=m, ascending=False).reset_index(drop=True)
        ranked['순위'] = ranked.index + 1
        print(f'\n▶ [{m} 순위]')
        display(ranked[['순위', company_col, period_col, m]])
        
    # 직전 기간 대비 성장률 비교표
    rate_cols = [company_col, period_col] + [f'{m}_증감률(%)' for m in metrics]
    print('\n--- [최신 기간 직전 대비 증감률(%) 비교] ---')
    display(latest_df[rate_cols].sort_values(by=f'{metrics[0]}_증감률(%)', ascending=False))
    
    return latest_df

df_latest = compare_companies(df_analyzed)

 [6] 최신 기간 기업별 재무 지표 비교

--- [지표별 기업 순위] ---

▶ [매출액 순위]

▶ [영업이익 순위]

▶ [순이익 순위]

▶ [자산 순위]

--- [최신 기간 직전 대비 증감률(%) 비교] ---


,기업명,기간,매출액,영업이익,순이익,자산
0,LG에너지솔루션,2024,256200,5800,3400,432000
1,SK하이닉스,2024,661900,234600,198000,1150000
2,삼성전자,2024,3050400,358000,325000,4800000
3,현대자동차,2024,1752300,165000,131000,2920000


,순위,기업명,기간,매출액
0,1,삼성전자,2024,3050400
1,2,현대자동차,2024,1752300
2,3,SK하이닉스,2024,661900
3,4,LG에너지솔루션,2024,256200


,순위,기업명,기간,영업이익
0,1,삼성전자,2024,358000
1,2,SK하이닉스,2024,234600
2,3,현대자동차,2024,165000
3,4,LG에너지솔루션,2024,5800


,순위,기업명,기간,순이익
0,1,삼성전자,2024,325000
1,2,SK하이닉스,2024,198000
2,3,현대자동차,2024,131000
3,4,LG에너지솔루션,2024,3400


,순위,기업명,기간,자산
0,1,삼성전자,2024,4800000
1,2,현대자동차,2024,2920000
2,3,SK하이닉스,2024,1150000
3,4,LG에너지솔루션,2024,432000


,기업명,기간,매출액_증감률(%),영업이익_증감률(%),순이익_증감률(%),자산_증감률(%)
1,SK하이닉스,2024,102.01,403.48,316.69,16.87
2,삼성전자,2024,17.81,445.15,109.85,5.28
3,현대자동차,2024,7.73,9.08,6.75,6.35
0,LG에너지솔루션,2024,-24.08,-73.19,-79.24,-4.84


## 7. SQL 담당자 결과와 비교 및 교차 검증

SQL 담당자가 산출한 결과와 지표별 값을 Outer Merge하여 불일치 항목 및 차이(`Pandas - SQL`)를 진단합니다.

In [8]:
def compare_with_sql_results(pandas_df, sql_df, company_col=COL_COMPANY, period_col=COL_PERIOD, metrics=METRIC_COLS, column_mapping=None):
    """SQL 산출물과 Pandas 산출물을 비교 검증하는 함수"""
    if sql_df is None:
        print('[알림] SQL 결과 데이터가 제공되지 않아 비교 검증을 생략합니다.')
        return None
    
    sql_clean = sql_df.copy()
    if column_mapping:
        sql_clean = sql_clean.rename(columns=column_mapping)
        
    sql_clean[period_col] = sql_clean[period_col].astype(str)
    for m in metrics:
        sql_clean[m] = pd.to_numeric(sql_clean[m], errors='coerce')
        
    keys = [company_col, period_col]
    merged = pd.merge(
        pandas_df[keys + metrics],
        sql_clean[keys + metrics],
        on=keys,
        how='outer',
        suffixes=('_Pandas', '_SQL'),
        indicator=True
    )
    
    # 차이 컬럼 생성
    for m in metrics:
        merged[f'{m}_차이(Pandas-SQL)'] = merged[f'{m}_Pandas'] - merged[f'{m}_SQL']
        
    print('=' * 60)
    print(' [7] SQL vs Pandas 결과 비교 검증')
    print('=' * 60)
    display(merged.sort_values(by=keys))
    
    diff_cols = [f'{m}_차이(Pandas-SQL)' for m in metrics]
    abs_diff = merged[diff_cols].abs().sum()
    print('\n--- [지표별 절대 오차 합계] ---')
    print(abs_diff)
    
    if (abs_diff == 0).all():
        print('✅ [검증 결과] 모든 지표의 값이 SQL 결과와 100% 완벽히 일치합니다.')
    else:
        print('⚠️ [검증 결과] 일부 지표에서 차이가 발생했습니다. 테이블 상세 내역을 확인하세요.')
        
    return merged

df_sql_comparison = compare_with_sql_results(df_analyzed, df_sql_raw)

 [7] SQL vs Pandas 결과 비교 검증

--- [지표별 절대 오차 합계] ---
매출액_차이(Pandas-SQL)     0
영업이익_차이(Pandas-SQL)    0
순이익_차이(Pandas-SQL)     0
자산_차이(Pandas-SQL)      0
dtype: int64
✅ [검증 결과] 모든 지표의 값이 SQL 결과와 100% 완벽히 일치합니다.


,기업명,기간,매출액_Pandas,영업이익_Pandas,순이익_Pandas,자산_Pandas,매출액_SQL,영업이익_SQL,순이익_SQL,자산_SQL,_merge,매출액_차이(Pandas-SQL),영업이익_차이(Pandas-SQL),순이익_차이(Pandas-SQL),자산_차이(Pandas-SQL)
0,LG에너지솔루션,2021,178519,7685,9298,237641,178519,7685,9298,237641,both,0,0,0,0
1,LG에너지솔루션,2022,255986,12132,7751,382994,255986,12132,7751,382994,both,0,0,0,0
2,LG에너지솔루션,2023,337455,21632,16380,453982,337455,21632,16380,453982,both,0,0,0,0
3,LG에너지솔루션,2024,256200,5800,3400,432000,256200,5800,3400,432000,both,0,0,0,0
4,SK하이닉스,2021,429978,124103,96162,963854,429978,124103,96162,963854,both,0,0,0,0
5,SK하이닉스,2022,446481,70067,24389,1038715,446481,70067,24389,1038715,both,0,0,0,0
6,SK하이닉스,2023,327657,-77303,-91375,984024,327657,-77303,-91375,984024,both,0,0,0,0
7,SK하이닉스,2024,661900,234600,198000,1150000,661900,234600,198000,1150000,both,0,0,0,0
8,삼성전자,2021,2796048,516339,399074,4266212,2796048,516339,399074,4266212,both,0,0,0,0
9,삼성전자,2022,3022314,433766,556541,4484245,3022314,433766,556541,4484245,both,0,0,0,0


## 8. 데이터 기반 객관적 결과 해석 (요구사항 7)

> **[원칙 준수]** 관측된 수치적 사실에만 기초하여 작성하며, 원천 데이터에 명시되지 않은 원인은 일체 추측하지 않습니다.

In [9]:
def generate_fact_based_insights(latest_df, company_col=COL_COMPANY, period_col=COL_PERIOD, metrics=METRIC_COLS):
    """추측을 배제하고 관측된 수치 사실만 자동 생성하는 해석 함수"""
    print('=' * 75)
    print('                  [데이터 기반 객관적 분석 결과 해석]                 ')
    print('=' * 75)
    
    # 1. 기업별 최신 기간 증감 사실
    print('\n1. [기업별 최신 기간 직전 대비 변동 사실]')
    for _, row in latest_df.iterrows():
        c = row[company_col]
        p = row[period_col]
        print(f'\n▶ {c} ({period_col}: {p})')
        for m in metrics:
            val = row[m]
            diff = row[f'{m}_증감액']
            rate = row[f'{m}_증감률(%)']
            
            if pd.isna(diff):
                print(f'  - {m}: 당기 {val:,.2f} (직전 비교 데이터 없음)')
            elif pd.isna(rate):
                print(f'  - {m}: 당기 {val:,.2f} / 증감액 {diff:+,.2f} (전기 기준값 0으로 증감률 산출 불가)')
            else:
                direction = '증가' if diff > 0 else ('감소' if diff < 0 else '변동 없음')
                print(f'  - {m}: 당기 {val:,.2f} / 직전 대비 {direction} (증감액: {diff:+,.2f}, 증감률: {rate:+,.2f}%)')
                
    # 2. 기업 간 규모 순위 사실
    print('\n2. [최신 기간 기업 간 규모 비교 사실]')
    for m in metrics:
        top = latest_df.sort_values(by=m, ascending=False).iloc[0]
        bottom = latest_df.sort_values(by=m, ascending=True).iloc[0]
        print(f'- {m}: {top[company_col]}이(가) {top[m]:,.2f}로 가장 크고, {bottom[company_col]}이(가) {bottom[m]:,.2f}로 가장 작게 관측됨.')

generate_fact_based_insights(df_latest)

                  [데이터 기반 객관적 분석 결과 해석]                 

1. [기업별 최신 기간 직전 대비 변동 사실]

▶ LG에너지솔루션 (기간: 2024)
  - 매출액: 당기 256,200.00 / 직전 대비 감소 (증감액: -81,255.00, 증감률: -24.08%)
  - 영업이익: 당기 5,800.00 / 직전 대비 감소 (증감액: -15,832.00, 증감률: -73.19%)
  - 순이익: 당기 3,400.00 / 직전 대비 감소 (증감액: -12,980.00, 증감률: -79.24%)
  - 자산: 당기 432,000.00 / 직전 대비 감소 (증감액: -21,982.00, 증감률: -4.84%)

▶ SK하이닉스 (기간: 2024)
  - 매출액: 당기 661,900.00 / 직전 대비 증가 (증감액: +334,243.00, 증감률: +102.01%)
  - 영업이익: 당기 234,600.00 / 직전 대비 증가 (증감액: +311,903.00, 증감률: +403.48%)
  - 순이익: 당기 198,000.00 / 직전 대비 증가 (증감액: +289,375.00, 증감률: +316.69%)
  - 자산: 당기 1,150,000.00 / 직전 대비 증가 (증감액: +165,976.00, 증감률: +16.87%)

▶ 삼성전자 (기간: 2024)
  - 매출액: 당기 3,050,400.00 / 직전 대비 증가 (증감액: +461,045.00, 증감률: +17.81%)
  - 영업이익: 당기 358,000.00 / 직전 대비 증가 (증감액: +292,330.00, 증감률: +445.15%)
  - 순이익: 당기 325,000.00 / 직전 대비 증가 (증감액: +170,129.00, 증감률: +109.85%)
  - 자산: 당기 4,800,000.00 / 직전 대비 증가 (증감액: +240,940.00, 증감률: +5.28%)

▶ 현대자동차 (기간: 2024)
  - 매출액: 당기 1,752,300.00 / 

## 9. 분석 결과 저장

산출된 최종 지표(증감액, 증감률 포함)를 파일로 내보냅니다.

In [10]:
OUTPUT_PATH = Path('../outputs/financial_analysis_pandas.csv') if Path('../outputs').exists() else Path('outputs/financial_analysis_pandas.csv')
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

df_analyzed.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')
print(f'✅ [저장 완료] 분석 결과 파일: {OUTPUT_PATH.resolve()}')

✅ [저장 완료] 분석 결과 파일: C:\Users\user\Documents\One day project\outputs\financial_analysis_pandas.csv
